<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0; font-size: 2em;">🔍 Lab 04: Trace Your Travel Agent</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">
    Contoso Travel Workshop — Microsoft Foundry Agent Observability
  </p>
</div>

## 🎯 What We Learn

In this lab, we instrument the Contoso Travel agent with **OpenTelemetry tracing** — first to the **console** for local debugging, then to **Azure Monitor** for production observability. You see exactly what happens inside the agent: the user query, tool invocations, model calls, and response generation.

> **Think of tracing as a flight recorder for your agent.** Just like aircraft black boxes capture every instrument reading and pilot action, OpenTelemetry tracing records every operation — so when something goes wrong, you can replay exactly what happened and pinpoint the cause.

---


## 🧳 The Contoso Travel Story

The Contoso Travel multi-agent system from Lab 03b is working well in development. The Flight Agent finds real flights, the Hotel Agent recommends actual properties, and the Car Rental Agent pulls from live inventory. The team is excited — until the first bug report comes in. A tester reports: *"I asked for a budget hotel in Paris and got a luxury recommendation at $450/night. The agent said it was 'affordable.'"* You stare at the final response. It looks wrong, but **you have no idea what happened inside.** Did the Hotel Agent receive the wrong query? Did it search correctly but rank results poorly? Did the orchestrator pass the budget constraint? The multi-agent system is a black box.

### 🔴 The Problem You Face Right Now

- **You can't debug what you can't see.** As your agent system grows more complex — multiple agents, tool calls, handoffs — understanding *why* something went wrong becomes nearly impossible.
- You need to see every step: which agent was called, what tools were invoked, how long each step took, and what data flowed between components.
- Without this visibility, every bug is a guessing game.

### ✅ What This Lab Solves

This lab adds **OpenTelemetry tracing** to your travel agent:
 - tracing to the **console** for immediate local debugging — seeing spans, durations, and attributes right in your notebook,
 - connecting to **Azure Monitor** so traces flow to the Foundry portal's Tracing tab for production-grade observability.

By the end, when something goes wrong, you'll know exactly where and why.

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #533483; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 20px 0;">
  <h2 style="margin: 0;">Part A: Console Tracing</h2>
</div>

## 2. Setup

---


In [ ]:
# Load env vars and enable GenAI tracing flags
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

# Must be set BEFORE importing tracing modules.
# Enables Azure SDK's experimental GenAI span generation.
os.environ["AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING"] = "true"
# Captures full prompt/response content in traces.
# Disable in prod if content contains PII.
os.environ["OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT"] = "true"

print("✅ Environment variables configured for GenAI tracing")


In [ ]:
# Configure OpenTelemetry to trace Azure SDK calls
from azure.core.settings import settings
# Tell Azure SDK to emit OTel spans for HTTP calls
settings.tracing_implementation = "opentelemetry"

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter
from azure.ai.projects.telemetry import AIProjectInstrumentor

# ConsoleSpanExporter prints spans to stdout (dev only)
span_exporter = ConsoleSpanExporter()
# TracerProvider manages span creation and lifecycle
tracer_provider = TracerProvider()
# SimpleSpanProcessor exports each span synchronously;
# use BatchSpanProcessor in production for performance.
tracer_provider.add_span_processor(SimpleSpanProcessor(span_exporter))
trace.set_tracer_provider(tracer_provider)
tracer = trace.get_tracer("contoso-travel-tracing")

# Auto-instruments Azure AI SDK: agent calls, tool
# invocations, and model requests emit spans automatically.
AIProjectInstrumentor().instrument()

print("✅ Console tracing configured")
print("   Traces will appear in the output below each cell")

In [ ]:
# Connect to Microsoft Foundry (no allow_preview needed
# here — tracing works with GA features).
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print(f"✅ Connected to Microsoft Foundry")

## 3. Create and Trace a Travel Agent

---


In [ ]:
# Create a simple traced agent for observability demo
agent = project_client.agents.create_version(
    agent_name="contoso-travel-traced",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions="You are the Contoso Travel Concierge. Help customers with travel questions about destinations, flights, hotels, and car rentals.",
    ),
)
print(f"✅ Agent created: {agent.name} (v{agent.version})")

## 4. Run a Traced Travel Query

Run this cell and check the console output below — OpenTelemetry spans appear for each operation.

---


In [ ]:
# start_as_current_span creates a parent span; all SDK
# calls inside become child spans in the same trace.
with tracer.start_as_current_span("contoso-travel-query"):
    conversation = openai_client.conversations.create()

    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        input="What are the best destinations for a summer vacation in Europe?",
    )

    print(f"\n🤖 Agent Response:\n{response.output_text}")
# Check console output above for trace_id, span_id,
# parent_id, and attributes like token counts.

## 5. Interpret the Trace Output

The console output above contains trace spans like:

- **`contoso-travel-query`** — The parent span we created
  - **`conversations.create`** — Creating the conversation
  - **`responses.create`** — The model inference call

Each span includes:
- `trace_id` — Groups all spans in one trace
- `span_id` — Unique ID for this span
- `parent_id` — Links child spans to parents
- `attributes` — Metadata like model name, token counts, etc.

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2196f3; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;"><strong>💡 Tip:</strong> In a multi-agent workflow, you'd see nested spans for each agent invocation — making it easy to trace the full execution path.</div>

---


<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #533483; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 20px 0;">
  <h2 style="margin: 0;">Part B: Azure Monitor Tracing</h2>
</div>

## 6. Configure Azure Monitor Tracing

---


In [ ]:
# Shut down console tracer to avoid duplicate spans
tracer_provider.shutdown()

from azure.monitor.opentelemetry import configure_azure_monitor

# Fetches the App Insights connection string linked to this
# Foundry project — no manual config needed.
app_insights_connection_string = project_client.telemetry.get_application_insights_connection_string()

# Replaces the console exporter with Azure Monitor exporter.
# Sets up a new TracerProvider + BatchSpanProcessor that
# sends spans to Application Insights automatically.
configure_azure_monitor(connection_string=app_insights_connection_string)

# Must get a new tracer after reconfiguring the provider
tracer = trace.get_tracer("contoso-travel-tracing")

print("✅ Azure Monitor tracing configured")
print(f"   Traces will flow to Application Insights")
print(f"   View them in the Foundry portal → Tracing tab")

## 7. Run a Traced Travel Query (Azure Monitor)

This trace flows to the Foundry portal. Traces may take a few minutes to appear in Azure Monitor.

---


In [ ]:
# Run a traced query with Azure Monitor exporter
with tracer.start_as_current_span("contoso-travel-paris-query") as span:
    # Custom attributes enable filtering in Azure Monitor.
    # Use a consistent namespace (e.g., "travel.", "contoso.")
    # so you can query traces by destination or agent.
    span.set_attribute("travel.destination", "Paris")
    span.set_attribute("travel.query_type", "trip_planning")
    span.set_attribute("contoso.agent_name", agent.name)

    conversation2 = openai_client.conversations.create()

    response = openai_client.responses.create(
        conversation=conversation2.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        input="I want to plan a week-long trip to Paris. What should I know about costs, weather, and must-see attractions?",
    )

    print(f"🤖 Agent Response:\n{response.output_text}")
    print(f"\n📊 Trace sent to Azure Monitor!")
    # trace_id in hex — use this to find the trace in portal
    print(f"   Trace ID: {span.get_span_context().trace_id:032x}")

## 8. View Traces in the Foundry Portal

To view your traces:

1. Go to the [Microsoft Foundry portal](https://ai.azure.com)
2. Open your project
3. Click on the **Tracing** tab in the left navigation
4. You should see your traces listed with the span names we defined
5. Click on a trace to see the full span tree

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #9c27b0; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;"><strong>⏱️ Note:</strong> Traces may take 2-5 minutes to appear in Azure Monitor after execution.</div>

![Tracing Tab](https://learn.microsoft.com/en-us/azure/foundry/observability/concepts/media/trace-agent-concept/trace-view.png)

---


## 9. Custom Span Attributes

Custom attributes make traces more useful for filtering and analysis. Add travel-specific metadata to enable business-level querying.

---


In [ ]:
# Richer custom attributes for business-level filtering.
# These appear in Azure Monitor's customDimensions and
# can be queried with KQL in App Insights.
with tracer.start_as_current_span("contoso-travel-london-query") as span:
    span.set_attribute("travel.destination", "London")
    span.set_attribute("travel.query_type", "budget_inquiry")
    span.set_attribute("travel.customer_segment", "business")
    span.set_attribute("contoso.agent_name", agent.name)
    span.set_attribute("contoso.agent_version", str(agent.version))

    response = openai_client.responses.create(
        conversation=conversation2.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        input="What business-class travel options are there from New York to London?",
    )

    print(f"🤖 Agent Response:\n{response.output_text}")
    print(f"\n📊 Custom attributes added to trace:")
    print(f"   travel.destination = London")
    print(f"   travel.query_type = budget_inquiry")
    print(f"   travel.customer_segment = business")

## 10. Cleanup

---


In [ ]:
# Cleanup: delete remote resources and close clients
openai_client.conversations.delete(conversation_id=conversation.id)
openai_client.conversations.delete(conversation_id=conversation2.id)
print("✅ Conversations deleted")

project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print("✅ Agent deleted")

openai_client.close()
project_client.close()
credential.close()
print("✅ Clients closed")

## 11. Summary & Next Steps

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #2e8b57; padding: 18px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <h3 style="margin-top: 0; color: #1a1a1a;">✅ What We Accomplished</h3>
  <ul style="margin-bottom: 0;">
  <li>Configured <b>console tracing</b> for local debugging with <code>ConsoleSpanExporter</code> and <code>AIProjectInstrumentor</code></li>
  <li>Configured <b>Azure Monitor tracing</b> for production observability using <code>configure_azure_monitor</code></li>
  <li>Ran traced travel queries and inspected span output</li>
  </ul>
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #1565c0; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <strong>💡 Key Takeaway:</strong> Tracing gives you X-ray vision into your agent's execution. Console tracing is great for development, while Azure Monitor tracing is essential for production monitoring. Custom attributes let you filter traces by business-relevant dimensions (e.g., destination, customer segment).
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #ff9800; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <strong>➡️ Next:</strong> In <a href="lab-05-evaluation.ipynb">Lab 05</a>, we'll <b>evaluate</b> the Contoso Travel agent for quality and safety — ensuring it gives accurate, fluent, and safe travel advice!
</div>

---
